# Conversation Sentiment Analysis - Model Training

## Objective

This notebook focuses on training machine learning models for
conversation sentiment classification using preprocessed MELD data.

The workflow includes:

- Loading preprocessed data
- Feature extraction using TF-IDF
- Training multiple machine learning models
- Comparing model performance
- Identifying the best-performing model

In [1]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")

print("Model training libraries imported successfully!")

Model training libraries imported successfully!


In [2]:
# ==========================================
# 2. LOAD PREPROCESSED DATASETS
# ==========================================

from pathlib import Path
import pandas as pd

# Locate project directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Load datasets
train_df = pd.read_csv(DATA_DIR / "train_processed.csv")
dev_df = pd.read_csv(DATA_DIR / "dev_processed.csv")
test_df = pd.read_csv(DATA_DIR / "test_processed.csv")

print("Preprocessed datasets loaded successfully!")
print("-" * 45)

print("Training dataset:", train_df.shape)
print("Validation dataset:", dev_df.shape)
print("Testing dataset:", test_df.shape)

print("\nSentiment classes:", train_df["Sentiment"].unique())

Preprocessed datasets loaded successfully!
---------------------------------------------
Training dataset: (9989, 12)
Validation dataset: (1109, 12)
Testing dataset: (2610, 12)

Sentiment classes: <ArrowStringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str


In [3]:
# ==========================================
# 3. BAG OF WORDS (BoW) FEATURE EXTRACTION
# ==========================================

from sklearn.feature_extraction.text import CountVectorizer

# Initialize BoW vectorizer
bow_vectorizer = CountVectorizer(max_features=5000)

# Learn vocabulary and transform training text
X_train_bow = bow_vectorizer.fit_transform(
    train_df["Cleaned_Utterance"].fillna("")
)

# Transform validation and testing data using training vocabulary
X_dev_bow = bow_vectorizer.transform(
    dev_df["Cleaned_Utterance"].fillna("")
)

X_test_bow = bow_vectorizer.transform(
    test_df["Cleaned_Utterance"].fillna("")
)

print("Bag of Words feature extraction completed!")
print("-" * 45)

print("Training feature matrix:", X_train_bow.shape)
print("Validation feature matrix:", X_dev_bow.shape)
print("Testing feature matrix:", X_test_bow.shape)

print("Vocabulary size:", len(bow_vectorizer.vocabulary_))

Bag of Words feature extraction completed!
---------------------------------------------
Training feature matrix: (9989, 4249)
Validation feature matrix: (1109, 4249)
Testing feature matrix: (2610, 4249)
Vocabulary size: 4249


In [4]:
# ==========================================
# 4. VISUALIZE BAG OF WORDS FEATURES
# ==========================================

# Get vocabulary feature names
bow_features = bow_vectorizer.get_feature_names_out()

print("First 20 vocabulary words:")
print(bow_features[:20])

# Display first 5 messages as numerical vectors
bow_sample = pd.DataFrame(
    X_train_bow[:5, :20].toarray(),
    columns=bow_features[:20]
)

print("\nBag of Words representation of first 5 messages:")
display(bow_sample)

First 20 vocabulary words:
['aa' 'aaaah' 'aaargh' 'aaron' 'abandoned' 'abandoner' 'able' 'aboard'
 'abort' 'abott' 'absolutely' 'abysmal' 'academic' 'accent' 'accept'
 'access' 'accident' 'accidentally' 'accord' 'account']

Bag of Words representation of first 5 messages:


,aa,aaaah,aaargh,aaron,abandoned,abandoner,able,aboard,abort,abott,absolutely,abysmal,academic,accent,accept,access,accident,accidentally,accord,account
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# ==========================================
# 4.1 DISPLAY RELEVANT BAG OF WORDS FEATURES
# ==========================================

# Select first 5 message vectors
sample_matrix = X_train_bow[:5]

# Find feature columns that appear in these messages
relevant_indices = np.unique(sample_matrix.indices)

# Select relevant feature names
relevant_features = bow_features[relevant_indices]

# Create a readable dataframe
bow_sample = pd.DataFrame(
    sample_matrix.toarray(),
    columns=bow_features
)

# Display only words that appear in these messages
bow_sample = bow_sample[relevant_features]

# Add original text
bow_sample.insert(
    0,
    "Original Utterance",
    train_df["Utterance"].head(5).values
)

display(bow_sample)

,Original Utterance,also,bit,company,duty,full,hand,let,little,must,person,point,right,system,talk,transition
0,also I was the point person on my company’s tr...,1,0,1,0,0,0,0,0,0,1,1,0,1,0,1
1,You must’ve had your hands full.,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0
2,That I did. That I did.,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,So let’s talk a little bit about your duties.,0,1,0,1,0,0,1,1,0,0,0,0,0,1,0
4,My duties? All right.,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0


In [6]:
# ==========================================
# 5. TF-IDF FEATURE EXTRACTION
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Fit on training data and transform
X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_df["Cleaned_Utterance"].fillna("")
)

# Transform validation and test data
X_dev_tfidf = tfidf_vectorizer.transform(
    dev_df["Cleaned_Utterance"].fillna("")
)

X_test_tfidf = tfidf_vectorizer.transform(
    test_df["Cleaned_Utterance"].fillna("")
)

print("TF-IDF feature extraction completed!")
print("-" * 45)

print("Training matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_dev_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

print("\nVocabulary size:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF feature extraction completed!
---------------------------------------------
Training matrix: (9989, 5000)
Validation matrix: (1109, 5000)
Testing matrix: (2610, 5000)

Vocabulary size: 5000


In [7]:
# ==========================================
# 6. TF-IDF FEATURE VISUALIZATION
# ==========================================

# Get feature names
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Select first 5 messages
sample_tfidf = X_train_tfidf[:5].toarray()

# Create dataframe
tfidf_sample_df = pd.DataFrame(
    sample_tfidf,
    columns=tfidf_features
)

# Find features that actually have non-zero values
relevant_features = tfidf_sample_df.columns[
    (tfidf_sample_df != 0).any()
]

# Display only relevant features
tfidf_sample_df = tfidf_sample_df[relevant_features]

# Add original messages
tfidf_sample_df.insert(
    0,
    "Original Utterance",
    train_df["Utterance"].head(5).values
)

display(tfidf_sample_df.round(3))

,Original Utterance,also,bit,company,duty,full,hand,let,let talk,little,little bit,must,person,point,right,system,talk
0,also I was the point person on my company’s tr...,0.412,0.000,0.488,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.412,0.439,0.000,0.479,0.000
1,You must’ve had your hands full.,0.000,0.000,0.000,0.000,0.643,0.528,0.000,0.000,0.000,0.000,0.555,0.000,0.000,0.000,0.000,0.000
2,That I did. That I did.,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,So let’s talk a little bit about your duties.,0.000,0.391,0.000,0.479,0.000,0.000,0.272,0.457,0.287,0.407,0.000,0.000,0.000,0.000,0.000,0.294
4,My duties? All right.,0.000,0.000,0.000,0.906,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.423,0.000,0.000


In [8]:
# ==========================================
# 7. BOW VS TF-IDF COMPARISON
# ==========================================

print("Original message:")
print(train_df["Utterance"].iloc[0])

print("\nBag of Words vector:")
print(X_train_bow[0].toarray())

print("\nTF-IDF vector:")
print(X_train_tfidf[0].toarray())

print("\nBoW vocabulary size:", X_train_bow.shape[1])
print("TF-IDF vocabulary size:", X_train_tfidf.shape[1])

Original message:
also I was the point person on my company’s transition from the KL-5 to GR-6 system.

Bag of Words vector:
[[0 0 0 ... 0 0 0]]

TF-IDF vector:
[[0. 0. 0. ... 0. 0. 0.]]

BoW vocabulary size: 4249
TF-IDF vocabulary size: 5000


In [9]:
# ==========================================
# 8. MULTINOMIAL NAIVE BAYES MODEL
# ==========================================

from sklearn.naive_bayes import MultinomialNB

# Prepare target labels
y_train = train_df["Sentiment"]
y_dev = dev_df["Sentiment"]
y_test = test_df["Sentiment"]

# Initialize model
nb_model = MultinomialNB()

# Train model using TF-IDF features
nb_model.fit(X_train_tfidf, y_train)

print("Multinomial Naive Bayes trained successfully!")
print("-" * 45)

print("Training samples:", len(y_train))
print("Sentiment classes:", nb_model.classes_)

Multinomial Naive Bayes trained successfully!
---------------------------------------------
Training samples: 9989
Sentiment classes: ['negative' 'neutral' 'positive']


In [10]:
# ==========================================
# 9. NAIVE BAYES VALIDATION EVALUATION
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict sentiment on validation data
y_dev_pred_nb = nb_model.predict(X_dev_tfidf)

# Calculate accuracy
nb_accuracy = accuracy_score(y_dev, y_dev_pred_nb)

print("NAIVE BAYES VALIDATION RESULTS")
print("=" * 45)

print(f"Accuracy: {nb_accuracy:.4f} ({nb_accuracy * 100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_dev, y_dev_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_dev, y_dev_pred_nb))

NAIVE BAYES VALIDATION RESULTS
Accuracy: 0.5113 (51.13%)

Classification Report:
              precision    recall  f1-score   support

    negative       0.66      0.26      0.37       406
     neutral       0.48      0.91      0.63       470
    positive       0.52      0.14      0.22       233

    accuracy                           0.51      1109
   macro avg       0.56      0.44      0.41      1109
weighted avg       0.56      0.51      0.45      1109


Confusion Matrix:
[[104 285  17]
 [ 27 430  13]
 [ 27 173  33]]


In [11]:
# ==========================================
# 10. LOGISTIC REGRESSION MODEL
# ==========================================

from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train the model using TF-IDF features
lr_model.fit(X_train_tfidf, y_train)

print("Logistic Regression trained successfully!")
print("-" * 45)
print("Training samples:", len(y_train))
print("Sentiment classes:", lr_model.classes_)

Logistic Regression trained successfully!
---------------------------------------------
Training samples: 9989
Sentiment classes: ['negative' 'neutral' 'positive']


In [12]:
# ==========================================
# 11. LOGISTIC REGRESSION EVALUATION
# ==========================================

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict validation data
y_dev_pred_lr = lr_model.predict(X_dev_tfidf)

# Calculate accuracy
lr_accuracy = accuracy_score(y_dev, y_dev_pred_lr)

print("Logistic Regression Validation Accuracy:", lr_accuracy)
print("\nClassification Report:\n")
print(classification_report(y_dev, y_dev_pred_lr))

# Confusion Matrix
cm_lr = confusion_matrix(y_dev, y_dev_pred_lr)

print("\nConfusion Matrix:\n")
print(cm_lr)

Logistic Regression Validation Accuracy: 0.5121731289449954

Classification Report:

              precision    recall  f1-score   support

    negative       0.54      0.47      0.50       406
     neutral       0.55      0.60      0.57       470
    positive       0.39      0.41      0.40       233

    accuracy                           0.51      1109
   macro avg       0.49      0.49      0.49      1109
weighted avg       0.51      0.51      0.51      1109


Confusion Matrix:

[[191 135  80]
 [117 281  72]
 [ 45  92  96]]


In [13]:
# ==========================================
# 12. LINEAR SVM MODEL
# ==========================================

from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM trained successfully!")
print("-" * 45)
print("Training samples:", len(y_train))
print("Sentiment classes:", svm_model.classes_)

Linear SVM trained successfully!
---------------------------------------------
Training samples: 9989
Sentiment classes: ['negative' 'neutral' 'positive']


In [14]:
# ==========================================
# 13. LINEAR SVM EVALUATION
# ==========================================

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict validation data
y_dev_pred_svm = svm_model.predict(X_dev_tfidf)

# Calculate accuracy
svm_accuracy = accuracy_score(y_dev, y_dev_pred_svm)

print("Linear SVM Validation Accuracy:", svm_accuracy)

print("\nClassification Report:\n")
print(classification_report(y_dev, y_dev_pred_svm))

# Confusion Matrix
cm_svm = confusion_matrix(y_dev, y_dev_pred_svm)

print("\nConfusion Matrix:\n")
print(cm_svm)

Linear SVM Validation Accuracy: 0.48692515779981965

Classification Report:

              precision    recall  f1-score   support

    negative       0.53      0.40      0.46       406
     neutral       0.52      0.63      0.57       470
    positive       0.35      0.35      0.35       233

    accuracy                           0.49      1109
   macro avg       0.47      0.46      0.46      1109
weighted avg       0.49      0.49      0.48      1109


Confusion Matrix:

[[164 163  79]
 [103 295  72]
 [ 42 110  81]]


In [15]:
# ==========================================
# 14. MODEL COMPARISON
# ==========================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

models_comparison = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "Linear SVM"],
    "Accuracy": [
        accuracy_score(y_dev, y_dev_pred_nb),
        accuracy_score(y_dev, y_dev_pred_lr),
        accuracy_score(y_dev, y_dev_pred_svm)
    ],
    "Macro F1": [
        f1_score(y_dev, y_dev_pred_nb, average="macro"),
        f1_score(y_dev, y_dev_pred_lr, average="macro"),
        f1_score(y_dev, y_dev_pred_svm, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_dev, y_dev_pred_nb, average="weighted"),
        f1_score(y_dev, y_dev_pred_lr, average="weighted"),
        f1_score(y_dev, y_dev_pred_svm, average="weighted")
    ]
})

models_comparison = models_comparison.sort_values(
    by="Macro F1", ascending=False
)

models_comparison.round(4)

,Model,Accuracy,Macro F1,Weighted F1
1,Logistic Regression,0.5122,0.4924,0.5117
2,Linear SVM,0.4869,0.4585,0.4820
0,Naive Bayes,0.5113,0.4084,0.4502


In [16]:
# ==========================================
# 15. SAVE CLASSICAL MODELS FOR FRONTEND
# ==========================================

import joblib
from pathlib import Path

# Locate project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Create save directory
CLASSICAL_MODEL_DIR = PROJECT_ROOT / "models" / "classical"
CLASSICAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save TF-IDF vectorizer and trained models together
classical_models = {
    "tfidf_vectorizer": tfidf_vectorizer,
    "naive_bayes": nb_model,
    "logistic_regression": lr_model,
    "linear_svm": svm_model,
    "classes": ["Negative", "Neutral", "Positive"]
}

MODEL_FILE = CLASSICAL_MODEL_DIR / "classical_models.joblib"

joblib.dump(classical_models, MODEL_FILE)

print("Classical models saved successfully!")
print("=" * 45)
print("Saved location:", MODEL_FILE)
print("Models:", list(classical_models.keys()))

Classical models saved successfully!
Saved location: c:\Users\sansm\Desktop\NLP PROJECT CHATGPT\Conversation_Sentiment_Analysis\models\classical\classical_models.joblib
Models: ['tfidf_vectorizer', 'naive_bayes', 'logistic_regression', 'linear_svm', 'classes']
